#### Download da página HTML com requests

In [3]:
import requests
url = "https://store.steampowered.com/?l=portuguese"

try:
    resposta = requests.get(url)
    resposta.raise_for_status()

    with open("pagina_original.html", "w", encoding="utf-8") as arquivo:
        arquivo.write(resposta.text)

        print("Pagina slava com sucesso como 'pagina_original.html'.")

except requests.exceptions.RequestException as erro:
    print(f"Erro ao acessar a pagina: {erro}")

Pagina slava com sucesso como 'pagina_original.html'.


#### Extração de tabela com BeautifulSoup

In [5]:
from bs4 import BeautifulSoup

try:
    with open("pagina_original.html", "r", encoding="utf-8") as arquivo:
        soup = BeautifulSoup(arquivo, "html.parser")

    tabela = soup.find("table")

    if tabela is None:
        raise Exception("Tabela não encontrada.")

    # Cabeçalhos
    cabecalhos = [th.get_text(strip=True) for th in tabela.find_all("th")]
    print("Cabeçalhos:")
    print(cabecalhos)

    linhas = tabela.find_all("tr")[1:]

    jogos = []

    for linha in linhas[:5]:
        colunas = linha.find_all("td")

        jogo = {}

        for i, coluna in enumerate(colunas):
            if i < len(cabecalhos):
                jogo[cabecalhos[i]] = coluna.get_text(strip=True)

        jogos.append(jogo)

    print("\nPrimeiros 5 jogos:")

    for jogo in jogos:
        print(jogo)

except Exception as erro:
    print("Erro:", erro)

Erro: Tabela não encontrada.


####  Construção de dicionário e uso de sets

In [15]:
coluna = "Nome"
dicionario_jogos = {}

for jogo in jogos:
    dicionario_jogos[jogo[coluna]] = jogo

valores_unicos = set()
duplicatas = []

for jogo in jogos:
    valor = jogo[coluna]

    if valor in valores_unicos:
        duplicatas.append(valor)
    else:
        valores_unicos.add(valor)

print("Total de registros:", len(jogos))
print("Total de valores únicos:", len(valores_unicos))
print("Duplicatas encontradas:", duplicatas)

Total de registros: 5
Total de valores únicos: 5
Duplicatas encontradas: []


#### Exportar para CSV usando módulo csv

In [17]:
import csv

try:
    dados = list(dicionario_jogos.values())
    cabecalhos = dados[0].keys()

    with open("dados_scraping.csv", "w", newline="", encoding="utf-8") as arquivo_csv:
        escritor = csv.DictWriter(arquivo_csv, fieldnames=cabecalhos)
        escritor.writeheader()
        escritor.writerows(dados)

    print("Arquivo 'dados_scraping.csv' salvo com sucesso!")

except Exception as erro:
    print(f"Erro ao salvar o arquivo CSV: {erro}")

Arquivo 'dados_scraping.csv' salvo com sucesso!


####  Exportar para JSON usando módulo json

In [19]:
import json

try:
    dados = list(dicionario_jogos.values())
    with open("dados_scraping.json", "w", encoding="utf-8") as arquivo_json:
        json.dump(dados, arquivo_json, ensure_ascii=False, indent=4)

    print("Arquivo 'dados_scraping.json' salvo com sucesso!")
    
    with open("dados_scraping.json", "r", encoding="utf-8") as arquivo_json:
        conteudo = json.load(arquivo_json)

    print("\nConteúdo do arquivo JSON:")
    print(conteudo)

except Exception as erro:
    print(f"Erro ao manipular o arquivo JSON: {erro}")

Arquivo 'dados_scraping.json' salvo com sucesso!

Conteúdo do arquivo JSON:
[{'Nome': 'Cyberpunk 2077', 'Preço': 'R$199,90', 'Desconto': '-50%'}, {'Nome': 'Stardew Valley', 'Preço': 'R$24,99', 'Desconto': '-20%'}, {'Nome': 'Hades', 'Preço': 'R$73,99', 'Desconto': '-30%'}, {'Nome': 'Celeste', 'Preço': 'R$36,99', 'Desconto': '-75%'}, {'Nome': 'Terraria', 'Preço': 'R$32,99', 'Desconto': '-50%'}]


#### Carregar CSV parcialmente com Pandas

In [21]:
import pandas as pd

df = pd.read_csv("dados_scraping.csv", usecols=["Nome", "Preço"])
print("Primeiras linhas:")
print(df.head())
df_ordenado = df.sort_values(by="Nome")

print("\nDataFrame ordenado:")
print(df_ordenado)

Primeiras linhas:
             Nome     Preço
0  Cyberpunk 2077  R$199,90
1  Stardew Valley   R$24,99
2           Hades   R$73,99
3         Celeste   R$36,99
4        Terraria   R$32,99

DataFrame ordenado:
             Nome     Preço
3         Celeste   R$36,99
0  Cyberpunk 2077  R$199,90
2           Hades   R$73,99
1  Stardew Valley   R$24,99
4        Terraria   R$32,99


####  Manipular JSON com Pandas

In [23]:
import pandas as pd

df = pd.read_json("dados_scraping.json")
df = df.rename(columns={
    "Nome": "Game",
    "Preço": "Preco",
    "Desconto": "Desconto"
})

df["Preco"] = (
    df["Preco"]
    .str.replace("R$", "", regex=False)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

acima_150 = df[df["Preco"] > 150]

abaixo_7990 = df[df["Preco"] < 79.90]

print("Games acima de R$150,00")
print(acima_150)

print("\nGames abaixo de R$79,90")
print(abaixo_7990)

Games acima de R$150,00
             Game  Preco Desconto
0  Cyberpunk 2077  199.9     -50%

Games abaixo de R$79,90
             Game  Preco Desconto
1  Stardew Valley  24.99     -20%
2           Hades  73.99     -30%
3         Celeste  36.99     -75%
4        Terraria  32.99     -50%


####  Gerar relatório em Excel



In [27]:
df.to_excel("relatorio_final.xlsx", index=False)

print("Arquivo 'relatorio_final.xlsx' salvo com sucesso!")

Arquivo 'relatorio_final.xlsx' salvo com sucesso!


#### Criar e consultar banco SQL com SQLAlchemy

In [29]:
import pandas as pd
from sqlalchemy import create_engine

df = None
engine = None
conn = None

try:
    df = pd.read_csv("dados_scraping.csv")
    print("Arquivo carregado com sucesso!")
except FileNotFoundError:
    print("Erro: arquivo 'dados_scraping.csv' não encontrado.")
except Exception as erro:
    print(f"Erro ao carregar o CSV: {erro}")

try:
    engine = create_engine("sqlite:///:memory:")
    conn = engine.connect()
    print("Banco SQLite criado com sucesso!")
except Exception as erro:
    print(f"Erro ao criar o banco: {erro}")

try:
    if df is not None:
        df.to_sql(
            "tabela_scraping",
            conn,
            if_exists="replace",
            index=False
        )
        print("Tabela gravada com sucesso!")
    else:
        print("Não foi possível gravar a tabela: DataFrame inexistente.")
except Exception as erro:
    print(f"Erro ao gravar no banco: {erro}")

try:
    if conn is not None:
        consulta = pd.read_sql_query(
            "SELECT * FROM tabela_scraping",
            conn
        )

        print("\nResultado da consulta:")
        print(consulta)
    else:
        print("Conexão indisponível.")
except Exception as erro:
    print(f"Erro na consulta SQL: {erro}")
    
try:
    if conn is not None:
        conn.close()
        print("\nConexão encerrada.")
except Exception:
    pass

print("\nPipeline executado.")

Arquivo carregado com sucesso!
Banco SQLite criado com sucesso!
Tabela gravada com sucesso!

Resultado da consulta:
             Nome     Preço Desconto
0  Cyberpunk 2077  R$199,90     -50%
1  Stardew Valley   R$24,99     -20%
2           Hades   R$73,99     -30%
3         Celeste   R$36,99     -75%
4        Terraria   R$32,99     -50%

Conexão encerrada.

Pipeline executado.


####  Tratamento de exceções em leitura de arquivos

In [31]:
import pandas as pd
from sqlalchemy import create_engine

csv_dados = """id,valor,categoria,descricao
1,150,credito,Deposito realizado no app
2,-180,debito,Pagamento do boleto
3,400,credito,Recebimento TED
4,-50,debito,Compra online
5,1020,credito,Salario mensal
6,-200,debito,Supermercado
"""

# ERRO 1
# pd.read_csv() espera o caminho de um arquivo ou um objeto semelhante a arquivo.
# Como foi passada uma string comum, o Pandas tenta abrir um arquivo com esse nome.
# Exceção esperada: FileNotFoundError.
df = pd.read_csv(csv_dados)

print("Arquivo carregado!")

# ERRO 2
# A coluna "valor" só poderá ser acessada se o DataFrame tiver sido criado.
# Como a leitura anterior falha, esta linha gera UnboundLocalError ou NameError,
# pois "df" não existe.
media_valor = df["valor"].mean()

print("Média calculada:", media_valor)

# ERRO 3
# A coluna "origem" não existe no conjunto de dados.
# Caso o DataFrame existisse, esta linha geraria KeyError.
media_categoria = df["origem"].value_counts()

print("Contagem:", media_categoria)

# Esta parte normalmente não gera erro.
engine = create_engine("sqlite:///meubanco.db")
conn = engine.connect()

# ERRO 4
# Se "df" não existir, esta linha também falha
# (NameError ou UnboundLocalError).
# Se existir, grava normalmente.
df.to_sql("tabela_dados", conn, if_exists="replace", index=False)

print("Tabela gravada com sucesso!")

# ERRO 5
# Caso a tabela não tenha sido criada, a consulta SQL gera erro
# (OperationalError).
consulta = conn.execute("SELECT * FROM tabela_dados")

for linha in consulta:
    print(linha)

conn.close()

print("Pipeline executado com sucesso!")

FileNotFoundError: [Errno 2] No such file or directory: 'id,valor,categoria,descricao\n1,150,credito,Deposito realizado no app\n2,-180,debito,Pagamento do boleto\n3,400,credito,Recebimento TED\n4,-50,debito,Compra online\n5,1020,credito,Salario mensal\n6,-200,debito,Supermercado\n'

#### Diagnóstico do código

##### Erro 1:
FileNotFoundError

Causa:
O método pd.read_csv() recebeu uma string contendo os dados do CSV.
O Pandas interpretou essa string como o caminho de um arquivo, que não existe.

Próximos erros que ocorreriam caso o primeiro fosse corrigido:

- NameError/UnboundLocalError: o DataFrame df não seria criado.
- KeyError: a coluna "origem" não existe.
- NameError/UnboundLocalError: tentativa de gravar um DataFrame inexistente no banco.
- OperationalError: consulta em uma tabela SQL que pode não ter sido criada.

 #### Tratamento avançado: TypeError, ValueError e KeyError

In [33]:
import pandas as pd
from io import StringIO

csv_dados = """id,valor,categoria,descricao
1,150,credito,Deposito realizado no app
2,-180,debito,Pagamento do boleto
3,400,credito,Recebimento TED
4,-50,debito,Compra online
5,1020,credito,Salario mensal
6,-200,debito,Supermercado
"""

try:
    df = pd.read_csv(StringIO(csv_dados))
    print("Arquivo carregado com sucesso!")
except Exception as erro:
    print(f"Erro ao carregar os dados: {erro}")
    df = None
if df is not None:
    print("\nPrimeiras linhas:")
    print(df.head())

if df is not None:
    try:
        media_valor = df["valor"].mean()
    except KeyError:
        print("A coluna 'valor' não existe.")
    else:
        print(f"\nMédia da coluna valor: {media_valor}")

if df is not None:
    try:
        media_categoria = df["origem"].value_counts()
        print(media_categoria)
    except KeyError:
        print("\nA coluna 'origem' não existe e será ignorada.")
        pass

Arquivo carregado com sucesso!

Primeiras linhas:
   id  valor categoria                  descricao
0   1    150   credito  Deposito realizado no app
1   2   -180    debito        Pagamento do boleto
2   3    400   credito            Recebimento TED
3   4    -50    debito              Compra online
4   5   1020   credito             Salario mensal

Média da coluna valor: 190.0

A coluna 'origem' não existe e será ignorada.


#### Corrigir a etapa SQL e garantir execução completa

In [35]:
import pandas as pd
from sqlalchemy import create_engine
from io import StringIO

csv_dados = """id,valor,categoria,descricao
1,150,credito,Deposito realizado no app
2,-180,debito,Pagamento do boleto
3,400,credito,Recebimento TED
4,-50,debito,Compra online
5,1020,credito,Salario mensal
6,-200,debito,Supermercado
"""

# ===========================
# Leitura do CSV
# ===========================
try:
    df = pd.read_csv(StringIO(csv_dados))
    print("Arquivo carregado com sucesso!")
except Exception as erro:
    print(f"Erro ao carregar o CSV: {erro}")
    df = None

if df is not None:
    print(df.head())

# ===========================
# Média da coluna valor
# ===========================
if df is not None:
    try:
        media = df["valor"].mean()
    except KeyError:
        print("Coluna 'valor' não encontrada.")
    else:
        print(f"Média da coluna valor: {media}")

# ===========================
# Coluna inexistente
# ===========================
if df is not None:
    try:
        print(df["origem"].value_counts())
    except KeyError:
        print("A coluna 'origem' será ignorada.")
        pass

# ===========================
# Conexão SQL
# ===========================
conn = None

try:
    engine = create_engine("sqlite:///:memory:")
    conn = engine.connect()
except Exception as erro:
    print(f"Erro ao criar a conexão: {erro}")

# ===========================
# Gravar tabela
# ===========================
if conn is not None and df is not None:
    try:
        df.to_sql(
            "tabela_dados",
            conn,
            if_exists="replace",
            index=False
        )
    except Exception as erro:
        print(f"Erro ao gravar a tabela: {erro}")
    else:
        print("Tabela gravada com sucesso!")

# ===========================
# Consulta SQL
# ===========================
if conn is not None:
    try:
        consulta = pd.read_sql_query(
            "SELECT * FROM tabela_dados",
            conn
        )

    except Exception as erro:
        print(f"Erro na consulta SQL: {erro}")

    else:
        print("\nResultado da consulta:")
        print(consulta)

    finally:
        conn.close()
        print("Conexão encerrada.")

print("\nPipeline corrigido e executado com sucesso!")

Arquivo carregado com sucesso!
   id  valor categoria                  descricao
0   1    150   credito  Deposito realizado no app
1   2   -180    debito        Pagamento do boleto
2   3    400   credito            Recebimento TED
3   4    -50    debito              Compra online
4   5   1020   credito             Salario mensal
Média da coluna valor: 190.0
A coluna 'origem' será ignorada.
Tabela gravada com sucesso!

Resultado da consulta:
   id  valor categoria                  descricao
0   1    150   credito  Deposito realizado no app
1   2   -180    debito        Pagamento do boleto
2   3    400   credito            Recebimento TED
3   4    -50    debito              Compra online
4   5   1020   credito             Salario mensal
5   6   -200    debito               Supermercado
Conexão encerrada.

Pipeline corrigido e executado com sucesso!


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=20e07645-96fa-4bd5-bbca-601067e24fc8' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>